# L4 39 — validate the matched-layout Qwen 3B link

Validates the sole reserved repair lineage without updating weights:

1. neutral held-out fidelity; and
2. matched-layout action fidelity on 384 snapshots excluding all 256 snapshots used to compare the earlier adapters.

The pilot is permitted only if trained KL beats both shuffled and random under the matched layout.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, pathlib, subprocess, torch
assert torch.cuda.is_available(), 'Select a GPU runtime first'
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True))
for name in ('HF_TOKEN', 'L4_RECEIVER_URL', 'L4_RECEIVER_TOKEN'):
    try:
        value = userdata.get(name)
        if value: os.environ[name] = value
    except Exception:
        print(f'{name}: not configured (receiver secrets are optional)')

In [ ]:
REPO = 'https://github.com/harrywinner2/rival-arena-capstone.git'
REVISION = '0d4e47a8b6b0554913e2f754c4b089312cd018b6'
WORK = pathlib.Path('/content/rival-arena-capstone')
if not WORK.exists(): subprocess.run(['git', 'clone', REPO, str(WORK)], check=True)
subprocess.run(['git', '-C', str(WORK), 'fetch', '--all'], check=True)
subprocess.run(['git', '-C', str(WORK), 'checkout', REVISION], check=True)
subprocess.run(['pip', 'install', '-q', '-r', str(WORK/'followup-representational/requirements.txt')], check=True)

In [ ]:
SOURCE_JOB_ID = 'faithful-qwen3b-t4-001'
JOB_ID = 'matched-qwen3b-t4-001'
MODEL = 'Qwen/Qwen2.5-3B-Instruct'
ROOT = pathlib.Path('/content/drive/MyDrive/rival-arena-l4')
SOURCE_DIR = ROOT/SOURCE_JOB_ID
JOB_DIR = ROOT/JOB_ID
MATCHES = SOURCE_DIR/'arena_ipd_confirmatory_v3/matches.jsonl'
EXCLUSIONS = SOURCE_DIR/'arena_context_fidelity_matched_v2/snapshots.jsonl'
for required in (JOB_DIR/'faithful_link.pt', MATCHES, EXCLUSIONS):
    assert required.exists(), f'Missing required Drive artifact: {required}'
print('Validating:', JOB_DIR)

In [ ]:
neutral = [
    'python', 'scripts/validate_link.py',
    '--model', MODEL,
    '--job-dir', JOB_DIR,
    '--job-id', JOB_ID,
    '--examples', '256',
    '--probe-examples', '240',
    '--generation-samples', '12',
]
print(' '.join(map(str, neutral)))
subprocess.run(list(map(str, neutral)), cwd=WORK/'followup-representational', check=True, env=os.environ)

In [ ]:
deployment = [
    'python', 'scripts/diagnose_arena_fidelity.py',
    '--model', MODEL,
    '--job-dir', JOB_DIR,
    '--job-id', JOB_ID,
    '--matches', MATCHES,
    '--exclude-snapshots', EXCLUSIONS,
    '--samples', '384',
    '--seed', '20260728',
    '--layout', 'matched',
]
print(' '.join(map(str, deployment)))
subprocess.run(list(map(str, deployment)), cwd=WORK/'followup-representational', check=True, env=os.environ)

## Completion

Expected outputs: `validation_report.json` and `arena_context_fidelity_matched_v2/` inside `MyDrive/rival-arena-l4/matched-qwen3b-t4-001/`. Do not launch an arena pilot until this report is reviewed.